In [159]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [160]:
df = pd.read_csv("../data/raw/taxi_trip_pricing.csv")


# --------------------
# 1. Basic Cleaning
# --------------------
# Drop rows where target is mis


In [161]:
df = df.dropna(subset=["Trip_Price"])

In [162]:
# Convert numeric columns

In [163]:
num_cols = [
    "Trip_Distance_km", "Passenger_Count",
    "Base_Fare", "Per_Km_Rate", "Per_Minute_Rate",
    "Trip_Duration_Minutes"
]

df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")



# --------------------
# 2. Handle Missing Values
# --------------------
# Numeric: fill with median


In [164]:

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Categorical: fill with "Unknown"

In [165]:
cat_cols = [
    "Time_of_Day", "Day_of_Week",
    "Traffic_Conditions", "Weather"
]


# --------------------
# 3. Feature Engineering
# --------------------


In [166]:
#cleaning
df = df[df["Trip_Distance_km"] < 100]
df = df[df["Trip_Price"] < df["Trip_Price"].quantile(0.99)]

#Price components
df["Distance_Cost"] = df["Trip_Distance_km"] * df["Per_Km_Rate"]
df["Time_Cost"] = df["Trip_Duration_Minutes"] * df["Per_Minute_Rate"]

# Speed feature
df["Avg_Speed"] = df["Trip_Distance_km"] / (df["Trip_Duration_Minutes"] / 60 + 1e-5)

# Peak indicator
df["Is_Peak"] = df["Time_of_Day"].isin(["Morning", "Evening"]).astype(int)

# Weekend flag
df["Is_Weekend"] = (df["Day_of_Week"] == "Weekend").astype(int)

# High traffic flag
df["High_Traffic"] = (df["Traffic_Conditions"] == "High").astype(int)

# Bad weather flag
df["Bad_Weather"] = df["Weather"].isin(["Rain", "Snow"]).astype(int)

#interaction features
df["Traffic_Time"] = df["High_Traffic"] * df["Trip_Duration_Minutes"]
df["Distance_per_Min"] = df["Trip_Distance_km"] / (df["Trip_Duration_Minutes"] + 1)
df["Fare_per_Km"] = df["Base_Fare"] / (df["Trip_Distance_km"] + 1)

#binning
df["Distance_Bin"] = pd.cut(df["Trip_Distance_km"], bins=5)




# --------------------
# 4. Prepare Features
# ------------------


In [167]:

target = "Trip_Price"

features = df.drop(columns=[target])
y = df[target]


# Updated column groups
num_features = features.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = features.select_dtypes(include=["object"]).columns.tolist()


C:\Users\3r7mef\AppData\Local\Temp\ipykernel_26384\127616141.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = features.select_dtypes(include=["object"]).columns.tolist()



# --------------------
# 5. Preprocessing Pipeline
# --------------------


In [168]:

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])

# Final pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor)
])

# Transform features
X = pipeline.fit_transform(features)



# --------------------
# 6. Output
# --------------------

In [169]:

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (921, 32)
Target shape: (921,)


# --------------------
# Save pipeline
# --------------------

In [170]:
from pathlib import Path
import joblib

# Save processed dataframe
output_path = Path("../data/prepped/taxi_trip_pricing_prepped.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

# Save pipeline
pipeline_path = Path("../models/preprocessing_pipeline.joblib")
pipeline_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, pipeline_path)

['..\\models\\preprocessing_pipeline.joblib']

# Train

In [171]:
import optuna
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

y_log = np.log1p(y)
# --------------------
# Split data
# --------------------
X_train, X_test, y_train, y_test = train_test_split(
    features,
    y_log,
    test_size=0.2,
    random_state=42
)

# --------------------
# Define objective function
# --------------------

def objective(trial):

    model = XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 200, 800),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 0.01, 10.0, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 0.01, 10.0, log=True),
        random_state=42
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Train
    pipeline.fit(X_train, y_train)

    # Predict (log scale)
    preds_log = pipeline.predict(X_test)

    # Convert to original scale
    preds = np.expm1(preds_log)
    y_test_original = np.expm1(y_test)

    # Compute RMSE in real units
    rmse = np.sqrt(mean_squared_error(y_test_original, preds))

    return rmse


# --------------------
# Run optimisation
# --------------------
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

# --------------------
# Best results
# --------------------
print("✅ Best RMSE:", study.best_value)
print("✅ Best params:", study.best_params)

[I 2026-05-08 15:08:06,457] A new study created in memory with name: no-name-8e26f01b-a27a-4642-aadb-99b87124a959
[I 2026-05-08 15:08:07,415] Trial 0 finished with value: 5.846467135299103 and parameters: {'n_estimators': 372, 'max_depth': 7, 'learning_rate': 0.020250599178136607, 'subsample': 0.9213983668134705, 'colsample_bytree': 0.7547581652711474, 'reg_lambda': 5.530085463724707, 'reg_alpha': 0.8040323182986178}. Best is trial 0 with value: 5.846467135299103.
[I 2026-05-08 15:08:08,605] Trial 1 finished with value: 6.064555362193992 and parameters: {'n_estimators': 327, 'max_depth': 10, 'learning_rate': 0.01551464803729927, 'subsample': 0.7223890713223882, 'colsample_bytree': 0.8005249197458387, 'reg_lambda': 0.173142745098711, 'reg_alpha': 1.0390184585335984}. Best is trial 0 with value: 5.846467135299103.
[I 2026-05-08 15:08:10,076] Trial 2 finished with value: 5.8286063414062355 and parameters: {'n_estimators': 646, 'max_depth': 9, 'learning_rate': 0.15041758555268883, 'subsamp

✅ Best RMSE: 5.038821877439527
✅ Best params: {'n_estimators': 206, 'max_depth': 3, 'learning_rate': 0.03744371106068879, 'subsample': 0.9818040877952982, 'colsample_bytree': 0.8713025861826588, 'reg_lambda': 3.4252901256390267, 'reg_alpha': 0.015500032227185902}


In [172]:
#save params
import json
from pathlib import Path

params_path = Path("../models/best_params.json")

params_path.parent.mkdir(exist_ok=True)

with open(params_path, "w") as f:
    json.dump(study.best_params, f, indent=4)

print("✅ Best params saved!")

✅ Best params saved!
